# DX 603: Project Milestone Two: Modeling and Feature Engineering

### Due: Sunday July 26 @ 11:59PM (with grace period of 2 hours & 1 minute)

### Overview

In Milestone 1, you explored the Zillow dataset, cleaned the data, and developed hypotheses about how preprocessing and feature engineering might improve predictive performance.

In this milestone, you will  develop, evaluate, and refine several machine learning models using those ideas. Rather than simply searching for the best algorithm, you will follow an iterative modeling workflow by:

1. Establishing baseline performance using several regression models.
2. Testing the preprocessing and feature engineering ideas proposed in Milestone 1.
3. Refining the feature set through feature selection.
4. Optimizing model performance through hyperparameter tuning.
5. Comparing the evolution of your models and selecting a final model to evaluate on the held-out test set.

Throughout this milestone, use **repeated 5-fold cross-validation (5 repeats)** to guide your modeling decisions. The held-out test set should be used only once, after all modeling decisions have been completed.




In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



## Prelude: Load Your Preprocessed Dataset from Milestone 1

In Milestone 1, you cleaned the Zillow dataset by removing unsuitable features, handling missing values, and encoding categorical variables. In this milestone, you will build, compare, and improve several regression models using that prepared dataset.

Begin by returning to your Milestone 1 notebook and rerunning your code through Part 3, where your dataset has been completely cleaned and encoded, but before any experimental feature engineering ideas were evaluated. Save this dataset and use it as the starting point for this milestone.

For example:

```python
# In Milestone 1
df_cleaned.to_csv("zillow_cleaned.csv", index=False)
```

```python
# In Milestone 2
df = pd.read_csv("zillow_cleaned.csv")
```

Next:

1. Separate the predictors (`X`) from the target (`y`).
2. Split the dataset into training and test sets using `train_test_split`.

Some regression models, such as **Ridge Regression** and **Lasso Regression**, require feature scaling. If you use one of these models, standardize the predictor variables **using only the training data**, then apply the same transformation to the test data.

```python
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

**Notes**

- Ordinary Linear Regression, Decision Trees, Random Forests, and HistGradientBoosting do **not** require feature scaling.
- If you create additional features later in this milestone and are using a scaled model, repeat the scaling step so the new features are transformed consistently.
- Throughout this milestone, use the same training/test split so that all models are evaluated on identical data.

In [2]:
# Add as many cells as you need
df = pd.read_csv("zillow_cleaned.csv")
df

,parcelid,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,finishedsquarefeet6,fireplacecnt,...,threequarterbathnbr,typeconstructiontypeid,unitcnt,yearbuilt,numberofstories,fireplaceflag,assessmentyear,taxdelinquencyflag,censustractandblock,taxvaluedollarcnt
0,14297519,0.0,3.5,4.0,NaN,3.5,3100.0,3100.0,0.0,0.0,...,1.0,NaN,NaN,1998.0,NaN,0,2016.0,NaN,6.059063e+13,1023282.0
1,17052889,0.0,1.0,2.0,NaN,1.0,1465.0,1465.0,0.0,1.0,...,0.0,NaN,NaN,1967.0,1.0,0,2016.0,NaN,6.111001e+13,464000.0
2,14186244,0.0,2.0,3.0,NaN,2.0,1243.0,1243.0,0.0,0.0,...,0.0,NaN,NaN,1962.0,1.0,0,2016.0,NaN,6.059022e+13,564778.0
3,12177905,0.0,3.0,4.0,8.0,3.0,2376.0,2376.0,0.0,0.0,...,0.0,NaN,1.0,1970.0,NaN,0,2016.0,NaN,6.037300e+13,145143.0
4,10887214,1.0,3.0,3.0,8.0,3.0,1312.0,1312.0,0.0,0.0,...,0.0,NaN,1.0,1964.0,NaN,0,2016.0,NaN,6.037124e+13,119407.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77608,10833991,1.0,3.0,3.0,8.0,3.0,1741.0,1741.0,0.0,0.0,...,0.0,NaN,1.0,1980.0,NaN,0,2016.0,NaN,6.037132e+13,379000.0
77609,11000655,0.0,2.0,2.0,6.0,2.0,1286.0,1286.0,0.0,0.0,...,0.0,NaN,1.0,1940.0,NaN,0,2016.0,NaN,6.037101e+13,354621.0
77610,17239384,0.0,2.0,4.0,NaN,2.0,1612.0,1612.0,0.0,1.0,...,0.0,NaN,NaN,1964.0,1.0,0,2016.0,NaN,6.111008e+13,67205.0
77611,12773139,1.0,1.0,3.0,4.0,1.0,1032.0,1032.0,0.0,0.0,...,0.0,NaN,1.0,1954.0,NaN,0,2016.0,NaN,6.037434e+13,49546.0


## Problem 1: Model Selection and Baselines [6 pts]

### 1.A Coding

Select **three** regression models from the following list and evaluate each one using the cleaned training dataset.

Use the default hyperparameters provided by scikit-learn (except where scaling is required).

Available models:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Decision Tree Regressor
* Bagging Regressor
* Random Forest Regressor
* HistGradientBoostingRegressor

For each of the three models you choose:

* Train using the **training dataset only**.
* Use **Repeated 5-Fold Cross-Validation** (5 repeats).
* Report validation performance:

  * Mean CV MAE
  * Standard Deviation of CV MAE

In [3]:
# Add as many code cells as needed.
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 

X = df.drop(columns=['taxvaluedollarcnt'])
y = df['taxvaluedollarcnt'].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_num = X.select_dtypes(include='number')


X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(X_num, y, test_size=0.2, random_state=42)
 
    
hgbr = HistGradientBoostingRegressor()
hgbr.fit(X_train_num,y_train_num)
hgbr_pred = hgbr.predict(X_test_num)

neg_MAE_scores = cross_val_score(hgbr, 
                                 X_train_num, y_train_num,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_hgbr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_hgbr = -np.std(neg_MAE_scores)

In [4]:
print(mean_cv_MAE_hgbr)
print(std_mae_hgbr)

196607.7376950603
-3151.1750636790134


In [8]:
from sklearn.ensemble import ExtraTreesRegressor
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
dtr = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42
)
X_sample = X_train_num.sample(5000, random_state=42) #testing if model works
y_sample = y_train_num.loc[X_sample.index]
dtr.fit(X_train_num,y_train_num)
#dtr_pred = dtr.predict(X_test_num)

neg_MAE_scores = cross_val_score(dtr, 
                                 X_train_num, y_train_num,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_dtr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_dtr = -np.std(neg_MAE_scores)

print(mean_cv_MAE_dtr)
print(std_mae_dtr)

197479.59821578872
-2445.0991630357066


In [10]:
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
rgr = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42
)
X_sample = X_train_num.sample(5000, random_state=42) #testing if model works
y_sample = y_train_num.loc[X_sample.index]
rgr.fit(X_sample,y_sample)
#dtr_pred = dtr.predict(X_test_num)

neg_MAE_scores = cross_val_score(rgr, 
                                 X_sample, y_sample,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_rgr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_rgr = -np.std(neg_MAE_scores)

print(mean_cv_MAE_rgr)
print(std_mae_rgr)

223511.3648231512
-18972.639046812154


In [12]:
from sklearn.metrics import mean_absolute_error,mean_squared_error #to test overfit vs underfit, hist boosting regression
train_pred = hgbr.predict(X_train_num)
train_MAE = mean_absolute_error(y_train_num, train_pred)

print("Train MAE:", train_MAE)
print("CV MAE:", mean_cv_MAE_hgbr)


Train MAE: 187633.97561423492
CV MAE: 196607.7376950603


In [13]:
train_pred = rgr.predict(X_train_num) #rgr 
train_MAE = mean_absolute_error(y_train_num, train_pred)

print("Train MAE:", train_MAE)
print("CV MAE:", mean_cv_MAE_hgbr)

Train MAE: 206264.4639696008
CV MAE: 196607.7376950603


In [14]:
train_pred = dtr.predict(X_train_num) #rgr 
train_MAE = mean_absolute_error(y_train_num, train_pred)

print("Train MAE:", train_MAE)
print("CV MAE:", mean_cv_MAE_hgbr)

Train MAE: 171967.14678323315
CV MAE: 196607.7376950603


### 1.B Discussion

Answer the following questions.

#### 1.B.1

Which of your three models achieved the **lowest validation MAE score **?

The Decision Tree Regressor had the lowest MAE score

#### 1.B.2

Which model produced the **smallest standard deviation** across the repeated cross-validation runs? What does this suggest about its stability?

The Decision Tree Regressor had the lowest STD as well. This likely suggests that this model is the most stable when compared to the actual trend of the data.


#### 1.B.3

Did any model appear to overfit or underfit? Explain your reasoning using the training and cross-validation results.

The random forest regressor and hist boosting regression both seemed to underfit due to its high training MAE compared to the decision tree regressor.

#### 1.B.4

Compare the overall strengths and weaknesses of the three models. Did any model consistently perform better, or were there important tradeoffs between accuracy and stability?

the decision tree regressor consistently performed the best, with its low STD and its lowest CV MAE score.

## Part 2: Evaluate Your Feature Engineering Hypotheses [6 pts]

### 2.A Coding

In **Milestone 1**, you proposed several preprocessing and feature engineering ideas that you believed might improve predictive performance.

Select **at least three** of those ideas and evaluate them.

These may include, for example:

* Creating new features
* Transforming existing features
* Removing features
* Combining features
* Other preprocessing ideas that you proposed in Milestone 1

For each idea:

* Apply the preprocessing or feature engineering to the **training dataset only**.
* Retrain the same three baseline models from **Problem 1** using repeated 5-fold cross-validation (5 repeats). 
* Compare the validation performance (mean CV MAE) and stability (standard deviation of CV MAE) with your original baseline results


> One of the most important things you can learn is that **not every clever idea results in an improvement**--they have to be evaluated by careful experiment.  And negative results are valuable if they are carefully evaluated and discussed!

In [5]:
# Add as many code cells as needed.

#idea 1: log transform on taxvaluedollarcnt and calcfinished squarefeet
# print(df['taxvaluedollarcnt'])
# print(df['calculatedfinishedsquarefeet'])
df['taxvaluedollarcnt'] = np.log1p(df['taxvaluedollarcnt'])
df['calculatedfinishedsquarefeet'] = np.log1p(df['calculatedfinishedsquarefeet'])
# print(df['taxvaluedollarcnt'])
# print(df['calculatedfinishedsquarefeet'])

repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 

X = df.drop(columns=['taxvaluedollarcnt'])
y = df['taxvaluedollarcnt'].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_num = X.select_dtypes(include='number')


X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(X_num, y, test_size=0.2, random_state=42)
 
    
hgbr = HistGradientBoostingRegressor()
hgbr.fit(X_train_num,y_train_num)
hgbr_pred = hgbr.predict(X_test_num)

neg_MAE_scores = cross_val_score(hgbr, 
                                 X_train_num, y_train_num,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_hgbr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_hgbr = -np.std(neg_MAE_scores)

In [7]:
print(mean_cv_MAE_hgbr)
print(std_mae_hgbr)

0.03572846543721728
-0.00024016472727848246


In [8]:
from sklearn.ensemble import ExtraTreesRegressor
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
dtr = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42
)
X_sample = X_train_num.sample(5000, random_state=42) #testing if model works
y_sample = y_train_num.loc[X_sample.index]
dtr.fit(X_train_num,y_train_num)
#dtr_pred = dtr.predict(X_test_num)

neg_MAE_scores = cross_val_score(dtr, 
                                 X_train_num, y_train_num,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_dtr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_dtr = -np.std(neg_MAE_scores)

print(mean_cv_MAE_dtr)
print(std_mae_dtr)

0.036216239344163
-0.00023599073002424073


In [9]:
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
rgr = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42
)
X_sample = X_train_num.sample(5000, random_state=42) #testing if model works
y_sample = y_train_num.loc[X_sample.index]
rgr.fit(X_sample,y_sample)
#dtr_pred = dtr.predict(X_test_num)

neg_MAE_scores = cross_val_score(rgr, 
                                 X_sample, y_sample,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_rgr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_rgr = -np.std(neg_MAE_scores)

print(mean_cv_MAE_rgr)
print(std_mae_rgr)

0.04273118645736131
-0.005385389009602049


In [13]:
#idea 2, capping outliers 

"""The Interquartile Range (IQR) is a statistical measure of spread that describes the middle 50% of a dataset"""

#refresh df
df = pd.read_csv("zillow_cleaned.csv")
def cap_outliers_iqr(df, columns, k=1.5):
    df_capped = df.copy()
    for col in columns:
        Q1 = df_capped[col].quantile(0.25)#pick two bounds to look out for and base the capping off of
        Q3 = df_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - k * IQR
        upper = Q3 + k * IQR
        df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
    return df_capped

X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(
    X_num, y, test_size=0.2, random_state=42
)
numeric_cols = X_num.columns.tolist()
X_train_num_capped = cap_outliers_iqr(X_train_num, numeric_cols, k=1.5)
X_test_num_capped= cap_outliers_iqr(X_test_num, numeric_cols, k=1.5)

hgbr_capped = HistGradientBoostingRegressor()
hgbr_capped.fit(X_train_num_capped, y_train_num)
hgbr_pred_capped = hgbr_capped.predict(X_test_num_capped)

neg_MAE_scores_capped = cross_val_score(
    hgbr_capped,
    X_train_num_capped, y_train_num,
    scoring='neg_mean_absolute_error',
    cv=repeated_cv,
    n_jobs=-1
)

mean_cv_MAE_hgbr_capped = -np.mean(neg_MAE_scores_capped)
std_mae_hgbr_capped = -np.std(neg_MAE_scores_capped)

print(mean_cv_MAE_hgbr_capped)
print(std_mae_hgbr_capped)

0.03568663115372388
-0.0002485614725529291


In [14]:
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
dtr = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42
)
dtr.fit(X_train_num_capped,y_train_num)

neg_MAE_scores_capped = cross_val_score(
    dtr,
    X_train_num_capped, y_train_num,
    scoring='neg_mean_absolute_error',
    cv=repeated_cv,
    n_jobs=-1
)

mean_cv_MAE_hgbr_capped = -np.mean(neg_MAE_scores_capped)
std_mae_hgbr_capped = -np.std(neg_MAE_scores_capped)

print(mean_cv_MAE_hgbr_capped)
print(std_mae_hgbr_capped)

0.03613614843626237
-0.00024989628679662117


In [16]:
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
rgr = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42)
    
rgr.fit(X_train_num_capped,y_train_num)

neg_MAE_scores_capped = cross_val_score(
    rgr,
    X_train_num_capped, y_train_num,
    scoring='neg_mean_absolute_error',
    cv=repeated_cv,
    n_jobs=-1
)

mean_cv_MAE_hgbr_capped = -np.mean(neg_MAE_scores_capped)
std_mae_hgbr_capped = -np.std(neg_MAE_scores_capped)

print(mean_cv_MAE_hgbr_capped)
print(std_mae_hgbr_capped)

0.03571982949261295
-0.0002469220782095835


In [20]:
#idea 3, dropping redundant features
df = pd.read_csv("zillow_cleaned.csv")
df = df.drop(columns=['bathroomcnt','finishedsquarefeet12'])
print(df.columns)
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 

X = df.drop(columns=['taxvaluedollarcnt'])
y = df['taxvaluedollarcnt'].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_num = X.select_dtypes(include='number')


X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(X_num, y, test_size=0.2, random_state=42)
 
    
hgbr = HistGradientBoostingRegressor()
hgbr.fit(X_train_num,y_train_num)
hgbr_pred = hgbr.predict(X_test_num)

neg_MAE_scores = cross_val_score(hgbr, 
                                 X_train_num, y_train_num,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_hgbr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_hgbr = -np.std(neg_MAE_scores)

print(mean_cv_MAE_hgbr)
print(std_mae_hgbr)

Index(['parcelid', 'airconditioningtypeid', 'bedroomcnt',
       'buildingqualitytypeid', 'calculatedbathnbr',
       'calculatedfinishedsquarefeet', 'finishedsquarefeet6', 'fireplacecnt',
       'fullbathcnt', 'garagecarcnt', 'garagetotalsqft', 'hashottuborspa',
       'heatingorsystemtypeid', 'lotsizesquarefeet', 'poolcnt', 'poolsizesum',
       'pooltypeid10', 'pooltypeid2', 'pooltypeid7',
       'propertycountylandusecode', 'propertylandusetypeid',
       'rawcensustractandblock', 'regionidcity', 'regionidcounty',
       'regionidzip', 'roomcnt', 'storytypeid', 'threequarterbathnbr',
       'typeconstructiontypeid', 'unitcnt', 'yearbuilt', 'numberofstories',
       'fireplaceflag', 'assessmentyear', 'taxdelinquencyflag',
       'censustractandblock', 'taxvaluedollarcnt'],
      dtype='object')
196293.6189118428
-2600.9031143435727


In [21]:
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
dtr = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42
)
X_sample = X_train_num.sample(5000, random_state=42) #testing if model works
y_sample = y_train_num.loc[X_sample.index]
dtr.fit(X_train_num,y_train_num)
#dtr_pred = dtr.predict(X_test_num)

neg_MAE_scores = cross_val_score(dtr, 
                                 X_train_num, y_train_num,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_dtr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_dtr = -np.std(neg_MAE_scores)

print(mean_cv_MAE_dtr)
print(std_mae_dtr)

196917.65878268774
-2462.4355550615956


In [22]:
repeated_cv = RepeatedKFold(n_splits     = 5, 
                            n_repeats    = 5,
                            random_state = 42
                           ) 
 
    
rgr = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,        # or None + max_leaf_nodes instead
    min_samples_leaf=5,  # prevents tiny leaves that bloat tree size
    n_jobs=1,             # see point 2
    random_state=42
)
X_sample = X_train_num.sample(5000, random_state=42) #testing if model works
y_sample = y_train_num.loc[X_sample.index]
rgr.fit(X_sample,y_sample)
#dtr_pred = dtr.predict(X_test_num)

neg_MAE_scores = cross_val_score(rgr, 
                                 X_sample, y_sample,
                                 scoring='neg_mean_absolute_error',            
                                 cv=repeated_cv,
                                 n_jobs=-1)

mean_cv_MAE_rgr = -np.mean(neg_MAE_scores)  # Convert negative MAE back to positive
std_mae_rgr = -np.std(neg_MAE_scores)

print(mean_cv_MAE_rgr)
print(std_mae_rgr)

224408.82680682727
-19901.493977948525


### 2.B Discussion

Answer the following questions.

#### 2.B.1

Which of your feature engineering ideas produced the largest improvement in validation performance?

The log transforms as well as the capping of outliers in the data both resulted in significant reductions of MAE and Stdev of the error, but the latter method provided a more consistent and slightly larger reduction in error. 

#### 2.B.2

Were any of your ideas unsuccessful or did they reduce model performance? Briefly explain.

All methods were successful due to them all reducing MAE and Stdev, though the dropping of redundant columns provided a very small reduction. Nevertheless it never raised the error.

#### 2.B.3

Did some models benefit more from feature engineering than others? If so, why do you think this occurred?

The log transformations and the capping of outliers massively benefitted the model, likely due to their greater effect in streamlining inconsistencies in the data and its patterns. Dropping repetitive columns do not seem to affect the overall trend as much as the former inconsistencies are left untouched which more directly hurt the model's performance.

#### 2.B.4

Which preprocessing or feature engineering changes will you keep for the remainder of the milestone? Briefly justify your decision.

The first two methods will definetly be utilized in the remainder of the milestone. They provided an extremely fruitful reduction of error in the data.

## Part 3: Refine the Feature Set [6 pts]

### 3.A Coding

Using your dataset after completing **Part 2** (including any preprocessing and feature engineering changes you decided to keep):

Investigate whether **feature selection** can further improve model performance.

You may use one or more of the following methods:

* Forward Selection (for linear regression models)
* Backward Selection (for linear regression models)
* Feature importance from tree-based models (for decision trees, Random Forests, Bagging, and HistGradientBoosting)
* Another reasonable feature selection method

For each of your three models:

* Select a subset of features using an appropriate feature selection method.
* Retrain the model using only the selected features.
* Evaluate the model using the same repeated cross-validation procedure as before.
* Report the validation performance (the mean and standard deviation of the CV MAE).

> Not every model will necessarily benefit from feature selection. Choose methods that are appropriate for the models you selected. Negative results are valuable if they are carefully evaluated and discussed!

In [27]:
# Add as many code cells as needed.

### 3.B Discussion

#### 3.B.1

Did feature selection improve the validation performance of any of your models?

> Replace this text with your answer.

#### 3.B.2

Were there features that were consistently retained (or consistently removed) across multiple models?

> Replace this text with your answer.

#### 3.B.3

Were any of your engineered features selected as important? If so, what does this suggest about the hypotheses you developed in Milestone 1?

> Replace this text with your answer.

#### 3.B.4

After feature selection, did simpler models perform as well as—or better than—the models using the full feature set? Briefly discuss any tradeoffs you observed between model complexity and predictive performance.

> Replace this text with your answer.

> Your text here

## Part 4: Tune Your Models [8 pts]

### 4.A Coding

Using the three models developed in **Part 3** (including your final preprocessing, feature engineering, and feature selection decisions):

Investigate whether **hyperparameter tuning** can further improve model performance.

For each of your three models:

* Select one or more important hyperparameters to tune.
* Use one or more appropriate tuning methods. Consider first using validation curves (`sweep_parameter`) to identify a promising region or performance plateau, followed by a focused search using methods such as:

    * GridSearchCV
    * RandomizedSearchCV
    * Another reasonable hyperparameter search method

* Choose hyperparameter values based on the validation results. If several nearby values produce similar validation performance (a performance plateau), prefer **values near the beginning of the plateau,** since they often produce simpler models with nearly identical predictive performance.
* Retrain the model using those hyperparameters.
* Evaluate the tuned model using repeated 5-fold cross-validation (5 repeats). 
* Report the validation performance (**mean** and **standard deviation** of the CV MAE).


In [28]:
# Add as many code cells as needed.

### 4.B Discussion

Answer the following questions.

#### 4.B.1

Which hyperparameters had the greatest impact on model performance? Briefly explain.

> Replace this text with your answer.

#### 4.B.2

Did hyperparameter tuning substantially improve the performance of all three models, or only some of them?

> Replace this text with your answer.

#### 4.B.3

Which tuning method(s) did you use for each model? Briefly explain why you chose those methods.

> Replace this text with your answer.

#### 4.B.4

After tuning, how did the relative performance of your three models change? Did tuning affect which model appeared to perform best?

> Replace this text with your answer.

## Part 5: Final Model and Workflow Assessment [14 pts]

### 5.A Coding

Using the work completed in **Parts 1–4**:

Select your **best-performing model** and prepare your final modeling pipeline.

Your pipeline should include all preprocessing, feature engineering, feature selection, and hyperparameter tuning decisions that you chose to retain.

Evaluate your final model by:

* Training on the complete training dataset.
* Reporting the **mean** and **standard deviation** of the repeated cross-validation MAE.
* Evaluating the model on the held-out test set.
* Reporting the final test MAE.

In [29]:
# Add as many code cells as needed.

### 5.B Discussion

Answer the following questions.

#### 5.B.1

Compare the performance of your final model with its original baseline from **Part 1**. Which changes contributed the most to the improvement?

> Replace this text with your answer.

#### 5.B.2

Looking back at the hypotheses you proposed in **Milestone 1**, which were supported by your experimental results? Were any hypotheses disproved?

> Replace this text with your answer.

#### 5.B.3

Why did you select this model as your final model? Discuss both its predictive performance and any other considerations (such as stability, simplicity, or interpretability).

> Replace this text with your answer.

#### 5.B.4

What did you learn about your dataset and the machine learning process through this end-to-end modeling workflow? If you had additional time, what would you investigate next?

> Replace this text with your answer.